# MiniCPM5-2B — Uncensored + Bahasa Indonesia (Stage 1)

**Set these before running:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Persistence | **Files** |

**Add this repo** either way works — the next cell finds it:
- upload `minicpm-uncensored/` as a Kaggle Dataset and *Add input*, or
- `File → Upload` a zip and unzip to `/kaggle/working`.

Then **Run All**. Order matters: cell 4 installs a newer `transformers` for Heretic, so the
training cells must run before it.

Final artifact: `/kaggle/working/minicpm5-uncensored-id-Q4_K_M.gguf` (~1.56 GB).


In [ ]:
# 0. locate the repo (works whether it came from /kaggle/working or /kaggle/input)
import glob, os, subprocess
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

cands = []
cands += glob.glob("/kaggle/working/minicpm-uncensored")
cands += glob.glob("/kaggle/input/*/minicpm-uncensored")
cands += glob.glob("/kaggle/input/*/kaggle/01-cpt-id.py") and [
    os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/*/kaggle/01-cpt-id.py")]
cands += glob.glob("/kaggle/working/**/kaggle/01-cpt-id.py") and [
    os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/working/**/kaggle/01-cpt-id.py", recursive=True)]

REPO = next((c for c in cands if os.path.isfile(os.path.join(c, "kaggle", "01-cpt-id.py"))), None)
assert REPO, "repo not found: attach the minicpm-uncensored folder as a Kaggle Dataset or upload it"
print("REPO =", REPO)
print(subprocess.run(["ls", os.path.join(REPO, "kaggle")], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)


In [ ]:
# 1. training stack. Order matters: unsloth's pins must win over transformers.
!pip -q install "transformers==4.57.3" "torch==2.7.1" "torchvision==0.22.1"
!pip -q install "unsloth>=2026.5" peft trl datasets accelerate
import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())
print("transformers", transformers.__version__)


## 2. Continued pretraining — 200M tokens of Indonesian

1526 steps × 2 GPUs × 2 batch × 16 accum × 2048 tokens. The script re-execs itself under
`torchrun` because a bare `python` on Kaggle only sees GPU 0.

**Needs ~20 GPU-hours and will be cut off.** That is expected: rerun this cell next session
and it resumes from the newest `checkpoint-250`. Save the notebook version so the output
becomes an input for the next run.


In [ ]:
# 2. continued pretraining (resumable)
!python {REPO}/kaggle/01-cpt-id.py


## 3. Mixed SFT — 63k Indonesian + 27k English, one epoch

Uses `/kaggle/working/minicpm5-id-cpt` when step 2 produced it, else `openbmb/MiniCPM5-2B`.


In [ ]:
# 3. mixed SFT
!python {REPO}/kaggle/02-sft-mix.py


## 4. Abliteration (Heretic)

From a pinned git commit, **not** PyPI: release 1.4.0 stops and asks whether to save, which
breaks unattended runs. OOM? Rerun with `--quant bnb_4bit`, keep `n_trials` at 100.


In [ ]:
# 4. abliteration
!pip -q install -U "git+https://github.com/p-e-w/heretic@ecde93f1e012bb344ac45de5cbd1027275f8b136"
!python {REPO}/kaggle/03-abliterate.py


## 5. GGUF export (Q4_K_M, ~1.56 GB)

Also deletes the 5 GB fp16 intermediate — `/kaggle/working` is only 20 GB.


In [ ]:
# 5. GGUF export
!bash {REPO}/kaggle/04-gguf.sh
!ls -lh /kaggle/working/*.gguf


## 6. Verification gate

Exits non-zero if Indonesian refusals exceed 2/10. **Do not download a model that fails.**


In [ ]:
# 6. verify (Indonesian refusals must be <= 2/10)
!python {REPO}/kaggle/05-verify.py \
    --model /kaggle/working/minicpm5-uncensored-id-abliterated --all-sets


## 7. Download

Only `/kaggle/working/minicpm5-uncensored-id-Q4_K_M.gguf` is needed locally. On the Mac:

```bash
cp <downloaded>.gguf minicpm-uncensored/models/
./scripts/deploy.sh models/minicpm5-uncensored-id-Q4_K_M.gguf minicpm5-uncensored-id
python scripts/probe.py --model minicpm5-uncensored-id --all-sets
```
